In [6]:
from pathlib import Path
import sys
import json


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src').exists():
            return candidate
    return start.resolve()


ROOT = find_project_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f'Project root: {ROOT}')

Project root: /Users/zitqan/Mid_tern_soa/Event-Extraction


In [7]:
from src.adapter.auto_adapter import AutoAdapter

In [8]:
PREFERRED_DATASET = 'BKEE'  # 'BKEE' | 'MAVEN' | 'UNIFIED'
ALLOW_FALLBACK = False
USE_SMOKE_SAMPLE_WHEN_MISSING = True

# This sample only verifies that the BKEE -> unified pipeline runs.
# Real BKEE files, when present, always take priority over it.
BKEE_SMOKE_SAMPLE = {
    'id': 'bkee-smoke-1',
    'text': 'Ông Nam gặp bà Lan tại Hà Nội.',
    'events': [
        {
            'event_type': 'Contact.Meet',
            'trigger': {'text': 'gặp', 'start': 8, 'end': 11},
            'arguments': [
                {'role': 'Entity', 'text': 'Ông Nam', 'start': 0, 'end': 7},
                {'role': 'Entity', 'text': 'bà Lan', 'start': 12, 'end': 18},
                {'role': 'Place', 'text': 'Hà Nội', 'start': 23, 'end': 29},
            ],
        }
    ],
}


def _dataset_candidates(root: Path, dataset: str):
    mapping = {
        'BKEE': [
            root / 'data/raw/BKEE/train.jsonl',
            root / 'data/raw/BKEE/valid.jsonl',
            root / 'data/raw/BKEE/dev.jsonl',
            root / 'data/raw/BKEE/test.jsonl',
            root / 'data/raw/BKEE/train.json',
            root / 'data/raw/BKEE/valid.json',
            root / 'data/raw/BKEE/dev.json',
            root / 'data/raw/BKEE/test.json',
        ],
        'MAVEN': [
            root / 'data/raw/MAVEN-Arg/train.jsonl',
            root / 'data/raw/MAVEN-Arg/valid.jsonl',
            root / 'data/raw/MAVEN-Arg/test.jsonl',
        ],
        'UNIFIED': [
            root / 'data/unified/train.jsonl',
            root / 'data/unified/valid.jsonl',
            root / 'data/unified/test.jsonl',
        ],
    }
    return mapping[dataset]


def load_first_sample(root: Path, preferred_dataset: str, allow_fallback: bool = False):
    order = [preferred_dataset]
    if allow_fallback:
        order.extend([name for name in ('BKEE', 'MAVEN', 'UNIFIED') if name != preferred_dataset])

    checked = []
    for dataset_name in order:
        for path in _dataset_candidates(root, dataset_name):
            checked.append(path)
            if not path.exists():
                continue

            if path.suffix in {'.json', '.jsonl'}:
                with path.open('r', encoding='utf-8') as f:
                    first_line = f.readline()
                    remainder = f.read()
                try:
                    payload = json.loads(first_line + remainder)
                except json.JSONDecodeError:
                    # Official BKEE processed files are JSONL despite .json suffix.
                    payload = json.loads(first_line)
                if isinstance(payload, list) and payload:
                    return dataset_name, path, payload[0], checked
                if isinstance(payload, dict):
                    records = payload.get('data')
                    if isinstance(records, list) and records:
                        return dataset_name, path, records[0], checked
                    return dataset_name, path, payload, checked

    return None, None, None, checked


detected_dataset, source_path, sample, checked_paths = load_first_sample(
    ROOT,
    PREFERRED_DATASET,
    allow_fallback=ALLOW_FALLBACK,
)

if source_path is None and PREFERRED_DATASET == 'BKEE' and USE_SMOKE_SAMPLE_WHEN_MISSING:
    detected_dataset = 'BKEE_SMOKE_SAMPLE'
    sample = BKEE_SMOKE_SAMPLE
    print('No real BKEE file was found; using the built-in smoke sample.')
    print('Add train/valid/test.json or .jsonl under data/raw/BKEE to use real data.')
elif source_path is None:
    print(f'No sample found for preferred dataset: {PREFERRED_DATASET}')
    print('Checked paths:')
    for p in checked_paths:
        print('-', p)
else:
    print(f'Detected dataset: {detected_dataset}')
    print(f'Loaded sample from: {source_path}')

Detected dataset: BKEE
Loaded sample from: /Users/zitqan/Mid_tern_soa/Event-Extraction/data/raw/BKEE/train.json


In [9]:
if sample is None:
    adapted = None
    print('Skipped adapting because no sample is available yet.')
else:
    adapted = AutoAdapter().adapt(sample)
    print('Adapter output (unified JSON):')
    print(adapted.to_json())

Adapter output (unified JSON):
{
    "id": "train-00000",
    "text": "Ngày 26/11 , Bộ Tài chính khai trương giao diện mới Cổng thông tin điện tử Bộ Tài chính và Kho bạc Nhà nước .",
    "events": [
        {
            "event_type": "Start-org",
            "trigger": {
                "text": "khai trương",
                "span": [
                    26,
                    37
                ]
            },
            "arguments": [
                {
                    "role": "Agent",
                    "mentions": [
                        {
                            "text": "Bộ Tài chính",
                            "span": [
                                13,
                                25
                            ]
                        }
                    ]
                },
                {
                    "role": "Time",
                    "mentions": [
                        {
                            "text": "Ngày 26/11",
                  

In [10]:
if adapted is None:
    print('No adapted output yet. Add BKEE data, then run all cells again.')
elif adapted.events:
    first = adapted.events[0]
    print('First event type:', first.event_type)
    print('First trigger:', first.trigger)
    print('First arguments:', first.arguments)

    assert adapted.id, 'Missing sample id after adaptation'
    assert adapted.text, 'Missing text after adaptation'
    assert first.event_type, 'Missing event type after adaptation'
    assert first.trigger.text, 'Missing trigger text after adaptation'
    start, end = first.trigger.span
    assert adapted.text[start:end] == first.trigger.text, 'Trigger span does not match text'
    print('BKEE adapter smoke test: PASSED')
else:
    print('No events found in the loaded sample.')

First event type: Start-org
First trigger: Trigger(text='khai trương', span=(26, 37))
First arguments: [Argument(role='Agent', mentions=[{'text': 'Bộ Tài chính', 'span': (13, 25)}]), Argument(role='Time', mentions=[{'text': 'Ngày 26/11', 'span': (0, 10)}]), Argument(role='Org', mentions=[{'text': 'Cổng thông tin điện tử Bộ Tài chính và Kho bạc Nhà nước', 'span': (52, 107)}])]
BKEE adapter smoke test: PASSED
